# RAW DATA ANALYSIS PIPELINE
This notebook performs data auditing before cleaning/EDA.
Focus: Schema, Nulls, Duplicates, Data Integrity, Validation

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df_raw = pd.read_csv('./dataset/01_raw/billings.csv')
print('Shape:', df_raw.shape)
print('\nColumns:', len(df_raw.columns))
print('\nInfo:')
print(df_raw.info())
print('\nHead:')
print(df_raw.head())

C:\Users\Asus\AppData\Local\Temp\ipykernel_36028\2442191968.py:1: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('./dataset/01_raw/billings.csv')


Shape: (122082, 59)

Columns: 59

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      122082 non-null  object 
 1   Renewal_Month               122082 non-null  object 
 2   Connection_Net              8037 non-null    float64
 3   Connection_Qty              8037 non-null    float64
 4   Discount_Amount             13551 non-null   object 
 5   Sustainability_Score        122082 non-null  float64
 6   Total_Renewal_Score_New     122082 non-null  float64
 7   Starting_Connection_Net     8545 non-null    float64
 8   Starting_Connection_Qty     8545 non-null    float64
 9   Last_Years_Price            112992 non-null  float64
 10  Last_Years_Date_Paid        0 non-null       float64
 11  Auto_Renewal_Score          122082 non-null  int64  
 12  Status_Scores               1220

In [3]:
os.makedirs('./dataset/01_raw/ad_hoc', exist_ok=True)

## 1. Column Listing

In [4]:
df_cols = df_raw.columns
df_cols_df = pd.DataFrame(df_cols, columns=['Columns'])
df_cols_df.to_csv('./dataset/01_raw/ad_hoc/columns.csv', index=False)

## 2. Null Analysis

In [5]:
null_counts = df_raw.isnull().sum()
print(null_counts)

null_columns = df_raw.columns[df_raw.isnull().any()]
print('\nColumns with null values:', list(null_columns))

null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns']) 
null_cols_df.to_csv('./dataset/01_raw/ad_hoc/null_columns.csv', index=False)

Co_Ref                             0
Renewal_Month                      0
Connection_Net                114045
Connection_Qty                114045
Discount_Amount               108531
Sustainability_Score               0
Total_Renewal_Score_New            0
Starting_Connection_Net       113537
Starting_Connection_Qty       113537
Last_Years_Price                9090
Last_Years_Date_Paid          122082
Auto_Renewal_Score                 0
Status_Scores                      0
Anchoring_Score                    0
Tenure_Scores                      0
Proforma_Auto_Renewal          18052
Proforma_World_Pay_Token       18052
Proforma_Date                    304
Current_Anchorings                 0
Current_Anchor_List            25957
Payment_Timeframe              20856
Registration_Date               1018
Proforma_Account_Stage          9229
Proforma_Audit_Status           9229
Current_Auto_Renewal_Flag          0
Current_World_Pay_Token            0
Renewal_Score_At_Release         126
P

In [6]:
def high_null_columns(df, threshold=0.5):
    return [col for col in df.columns if df[col].isna().mean() > threshold]

print('\nHigh null columns (>50%):', high_null_columns(df_raw))


High null columns (>50%): ['Connection_Net', 'Connection_Qty', 'Discount_Amount', 'Starting_Connection_Net', 'Starting_Connection_Qty', 'Last_Years_Date_Paid']


## 3. Duplicate Analysis

In [7]:
duplicate_rows = df_raw.duplicated().sum()
print('Duplicate rows:', duplicate_rows)

df_raw[df_raw.duplicated()].to_csv('./dataset/01_raw/ad_hoc/duplicate_rows.csv', index=False)

Duplicate rows: 0


## 4. Data Type & Classification

In [8]:
def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return 'datetime'

    if pd.api.types.is_numeric_dtype(series):
        return 'numerical'

    if series.dtype == 'object':
        try:
            pd.to_datetime(series.dropna().head(10), errors='raise')
            return 'datetime'
        except:
            pass

        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return 'time'
        except:
            pass

        return 'categorical'

    return 'categorical'

## 5. Unique Values Representation

In [9]:
def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f'{series.min()} - {series.max()}'

    if num_unique <= 20:
        return unique_vals.tolist()

    return 'High Cardinality'

## 6. Data Quality Checks

In [10]:
def is_id_column(series):
    return series.nunique(dropna=True) == len(series)

def detect_mixed_types(series):
    return series.apply(type).nunique() > 1

def primary_key_check(series):
    return {
        'nulls': series.isna().sum(),
        'duplicates': series.duplicated().sum()
    }

## 7. Validation Summary

In [11]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            'column_name': col,
            'data_type': str(series.dtype),
            'column_type': classify_column(series),
            'null_count': series.isna().sum(),
            'null_percentage': round((series.isna().sum() / len(df)) * 100, 2),
            'num_unique': series.nunique(dropna=True),
            'cardinality_ratio': round(series.nunique(dropna=True) / len(df), 4),
            'unique_values': get_unique_representation(series),
            'top_values': series.value_counts(dropna=True).head(5).to_dict(),
            'is_possible_id': is_id_column(series),
            'has_mixed_types': detect_mixed_types(series),
            'constant_column': series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)

In [12]:
df_validation = create_data_validation_df(df_raw)
print(df_validation.head())

df_validation.to_csv('./dataset/01_raw/ad_hoc/data_validation_summary.csv', index=False)

C:\Users\Asus\AppData\Local\Temp\ipykernel_36028\4022170350.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_36028\4022170350.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_36028\4022170350.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_36028\4022170350.py:10: Use

       column_name data_type  column_type  null_count  null_percentage  \
0           Co_Ref    object  categorical           0             0.00   
1    Renewal_Month    object     datetime           0             0.00   
2   Connection_Net   float64    numerical      114045            93.42   
3   Connection_Qty   float64    numerical      114045            93.42   
4  Discount_Amount    object  categorical      108531            88.90   

   num_unique  cardinality_ratio  \
0       47826             0.3918   
1          49             0.0004   
2         161             0.0013   
3          48             0.0004   
4          11             0.0001   

                                       unique_values  \
0                                   High Cardinality   
1                                   High Cardinality   
2                                      55.0 - 4752.0   
3                                         1.0 - 48.0   
4  [35.00%, 20.00%, 30.00%, 40.00%, 10.00%, 50.00...   

 

## ✅ RAW ANALYSIS COMPLETE